# 上・下延長確認メール送付＆Salesforce取込

指定したタブの H〜N 列を、稼働レコード（`kado__c`）へ上書きします。レコードの特定には最終列の「レコードID」を使います。

## 出力元

[延長確認シートを開く](https://docs.google.com/spreadsheets/d/13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk/edit?gid=809815426#gid=809815426)

フォームのシート名は、タブ名です。上のリンクのタブは `202612` です。

## 反映する列

| 列 | シート | Salesforce |
|---|---|---|
| H | 上案件 / バックオフィスからの / 対応ストップ案件チェック | BO延長確認停止（上案件） |
| I | 下案件 / バックオフィスからの / 対応ストップ案件チェック | BO延長確認停止（下案件） |
| J | ㊤更新意思 | `uekoushinishi__c` |
| K | ㊦更新意思 | `shitakoushinishi__c` |
| L | 確定期間 | `kauteikikan__c` |
| M | 契約終了理由 | `keiyakusyuryoriyu__c` |
| N | 他案件契約終了日（㊦担当者入力） | ラベルに「他案件契約終了日」を含む項目 |

空欄は Salesforce 側も空に上書きします。

## メール

稼働の更新に成功した行だけ送ります。

- H が FALSE の行は、Apex `SendEmailbyextension` を `Send` で呼び出して上延長確認メールを送ります
- I が FALSE の行は、Apex `SendEmailbyextensionLower` を `Send` で呼び出して下延長確認メールを送ります

画面の「上延長確認メールの送信」「下延長確認メールの送信」と同じ処理です。

## 実行手順

1. シークレットに `SF_CLIENT_ID` / `SF_USERNAME` / `SF_PRIVATE_KEY` を入れる
2. シート名を確認する
3. セルを実行する（Ctrl + F9）
4. メール送付前の確認表で、企業名・技術者名と上・下の宛先を確認する
5. 「反映を実行する」がオフのときは、更新もメール送信もしません


In [ ]:
# @title 上・下延長確認メール送付＆Salesforce取込
_worksheet_name = "202612"  # @param {"type":"string"}
_apply_updates = True  # @param {type:"boolean"}

!pip -q install simple-salesforce gspread pandas pyjwt cryptography requests

import html
import json
import re
import time
from datetime import date, datetime

import jwt
import requests
import gspread
from google.auth import default
from google.colab import auth, userdata
from simple_salesforce import Salesforce

SPREADSHEET_ID = "13ILjnnpoFRCk01uO7rYZ3dRmTljByWvAMLTJT81hoSk"
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/" + SPREADSHEET_ID + "/edit"
RECORD_ID_LABEL = "レコードID"
UE_MAIL_APEX = "SendEmailbyextension"
SHITA_MAIL_APEX = "SendEmailbyextensionLower"
# H列から N列（0始まりで 7〜13）
HN_START = 7
HN_END = 14
COLUMN_SPECS = [
    {
        "key": "stop_ue",
        "header_contains": "上案件",
        "label": "BO延長確認停止（上案件）",
        "api_name": None,
    },
    {
        "key": "stop_shita",
        "header_contains": "下案件",
        "label": "BO延長確認停止（下案件）",
        "api_name": None,
    },
    {
        "key": "ue_ishi",
        "header_contains": "㊤更新意思",
        "label": "㊤更新意思",
        "api_name": "uekoushinishi__c",
    },
    {
        "key": "shita_ishi",
        "header_contains": "㊦更新意思",
        "label": "㊦更新意思",
        "api_name": "shitakoushinishi__c",
    },
    {
        "key": "kikan",
        "header_contains": "確定期間",
        "label": "確定期間",
        "api_name": "kauteikikan__c",
    },
    {
        "key": "riyu",
        "header_contains": "契約終了理由",
        "label": "契約終了理由",
        "api_name": "keiyakusyuryoriyu__c",
    },
    {
        "key": "tajian",
        "header_contains": "他案件契約終了日",
        "label": None,
        "label_contains": "他案件契約終了日",
        "api_name": None,
    },
]


def _norm(value):
    return str(value or "").replace("\n", "").replace(" ", "").replace("\u3000", "").strip()


def get_salesforce_access_token():
    print("Salesforce に接続しています...")
    sf_client_id = userdata.get("SF_CLIENT_ID")
    sf_username = userdata.get("SF_USERNAME")
    sf_private_key = userdata.get("SF_PRIVATE_KEY")
    if not sf_client_id or not sf_username or not sf_private_key:
        raise RuntimeError(
            "Salesforce シークレット (SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY) が見つかりません。"
        )
    normalized_key = str(sf_private_key).strip().replace("\\n", "\n")
    assertion = jwt.encode(
        {
            "iss": sf_client_id,
            "sub": sf_username,
            "aud": "https://login.salesforce.com",
            "exp": int(time.time()) + 180,
        },
        normalized_key,
        algorithm="RS256",
    )
    res = requests.post(
        "https://login.salesforce.com/services/oauth2/token",
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer",
            "assertion": assertion,
        },
        timeout=60,
    )
    if res.status_code != 200:
        raise RuntimeError(f"Salesforce 認証に失敗しました: {res.status_code} {res.text}")
    token_data = res.json()
    return token_data["access_token"], token_data["instance_url"]


def setup_salesforce():
    access_token, instance_url = get_salesforce_access_token()
    print(f"接続先: {instance_url}")
    return Salesforce(instance_url=instance_url, session_id=access_token, version="64.0")


def setup_google_sheets():
    print("Google スプレッドシートに接続しています...")
    auth.authenticate_user()
    creds, _ = default()
    return gspread.authorize(creds)


def _describe_fields(sf):
    described = sf.kado__c.describe()
    return described.get("fields") or []


def _find_field(described, spec):
    by_name = {field.get("name"): field for field in described if field.get("name")}
    api_name = spec.get("api_name")
    if api_name and api_name in by_name:
        return by_name[api_name]
    label = spec.get("label")
    if label:
        target = _norm(label)
        for field in described:
            if _norm(field.get("label")) == target:
                return field
    needle = spec.get("label_contains")
    if needle:
        target = _norm(needle)
        hits = [field for field in described if target in _norm(field.get("label"))]
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            names = "、".join(
                f"{field.get('label')} ({field.get('name')})" for field in hits
            )
            raise RuntimeError(f"「{needle}」に一致する項目が複数あります: {names}")
    label_text = label or spec.get("label_contains") or api_name
    hints = [
        f"{field.get('label')} ({field.get('name')})"
        for field in described
        if any(
            word in _norm(field.get("label"))
            for word in ("ストップ", "延長確認", "終了日", "更新意思", "確定期間")
        )
    ]
    hint = "、".join(hints[:12]) or "(近い項目なし)"
    raise RuntimeError(f"kado__c に項目が見つかりません: {label_text}。近い項目: {hint}")


def _action_error_message(errors):
    parts = []
    for error in errors or []:
        if isinstance(error, dict):
            parts.append(str(error.get("message") or error))
        else:
            parts.append(str(error))
    return " / ".join(part for part in parts if part)


def _invoke_extension_mail(sf, action_name, record_id):
    """画面の延長確認メール送信と同じ Apex を Send で呼ぶ。"""
    try:
        result = sf.restful(
            f"actions/custom/apex/{action_name}",
            method="POST",
            json={"inputs": [{"recordId": record_id, "actionType": "Send"}]},
        )
    except Exception as exc:
        content = getattr(exc, "content", None)
        return False, str(content or exc)
    if isinstance(result, dict):
        result = [result]
    if not result:
        return False, "Apex の応答が空です"
    item = result[0]
    if item.get("isSuccess"):
        output = item.get("outputValues") or {}
        to_address = output.get("toAddress") or ""
        subject = output.get("subject") or ""
        if to_address and subject:
            return True, f"{to_address} / {subject}"
        return True, to_address or subject or "送信しました"
    message = _action_error_message(item.get("errors")) or str(item)
    return False, message


def _send_extension_mails(sf, updates, success_ids):
    sent_ids = {record_id for record_id in success_ids if record_id}
    lines = []
    ue_ok = 0
    shita_ok = 0
    for item in updates:
        record_id = item["payload"]["Id"]
        if record_id not in sent_ids:
            continue
        if item["send_ue"]:
            ok, detail = _invoke_extension_mail(sf, UE_MAIL_APEX, record_id)
            if ok:
                ue_ok += 1
                lines.append(f"{item['row']}行目 上延長確認メールを送信しました: {detail}")
            else:
                lines.append(f"{item['row']}行目 上延長確認メールの送信に失敗しました: {detail}")
        if item["send_shita"]:
            ok, detail = _invoke_extension_mail(sf, SHITA_MAIL_APEX, record_id)
            if ok:
                shita_ok += 1
                lines.append(f"{item['row']}行目 下延長確認メールを送信しました: {detail}")
            else:
                lines.append(f"{item['row']}行目 下延長確認メールの送信に失敗しました: {detail}")
    return ue_ok, shita_ok, lines


def _parse_bool(value):
    if isinstance(value, bool):
        return value
    text = str(value or "").strip().lower()
    return text in {"true", "1", "yes", "y", "はい"}


def _parse_date(value):
    text = str(value or "").strip()
    if not text:
        return None
    text = text.replace("/", "-").replace(".", "-")
    head = text[:10]
    try:
        return datetime.strptime(head, "%Y-%m-%d").date().isoformat()
    except ValueError:
        pass
    parts = text.split("-")
    if len(parts) >= 3 and all(part.isdigit() for part in parts[:3]):
        year, month, day = (int(part) for part in parts[:3])
        return date(year, month, day).isoformat()
    raise ValueError(f"日付として解釈できません: {value}")


def _convert(value, field):
    field_type = field.get("type")
    text = "" if value is None else str(value).strip()
    if field_type == "boolean":
        return _parse_bool(value)
    if text == "":
        return None
    if field_type == "date":
        return _parse_date(text)
    if field_type == "datetime":
        parsed = _parse_date(text)
        return None if parsed is None else parsed + "T00:00:00.000Z"
    if field_type in {"double", "currency", "percent", "int"}:
        number = float(text.replace(",", ""))
        return int(number) if field_type == "int" else number
    return text


def _read_sheet(gc, worksheet_name):
    spreadsheet = gc.open_by_key(SPREADSHEET_ID)
    titles = [ws.title for ws in spreadsheet.worksheets()]
    if worksheet_name not in titles:
        raise RuntimeError(
            f"シート '{worksheet_name}' がありません。存在するタブ: {', '.join(titles)}"
        )
    worksheet = spreadsheet.worksheet(worksheet_name)
    rows = worksheet.get_all_values()
    if not rows:
        raise RuntimeError("シートにデータがありません。")
    return rows


def _column_index(header, label):
    target = _norm(label)
    for index, name in enumerate(header):
        if _norm(name) == target:
            return index
    raise RuntimeError(f"ヘッダ「{label}」がありません。")


def _build_updates(rows, described):
    header = rows[0]
    if len(header) < HN_END:
        raise RuntimeError(f"H〜N列がありません。列数: {len(header)}")
    record_id_index = _column_index(header, RECORD_ID_LABEL)
    engineer_index = _column_index(header, "技術者名")
    ue_company_index = _column_index(header, "上企業名")
    shita_company_index = _column_index(header, "下企業名")
    resolved = []
    for offset, spec in enumerate(COLUMN_SPECS):
        index = HN_START + offset
        header_name = header[index]
        if spec["header_contains"] not in _norm(header_name):
            print(f"注意: {chr(65 + index)}列のヘッダが想定と違います: {header_name!r}")
        field = _find_field(described, spec)
        resolved.append((index, field))
        shown = str(header_name).replace("\n", " / ")
        print(f"{chr(65 + index)}列 {shown} → {field.get('label')} ({field.get('name')})")

    print(f"上延長確認メール: Apex {UE_MAIL_APEX}")
    print(f"下延長確認メール: Apex {SHITA_MAIL_APEX}")

    updates = []
    skipped = 0
    ue_mail_count = 0
    shita_mail_count = 0
    ue_stop_name = resolved[0][1]["name"]
    shita_stop_name = resolved[1][1]["name"]
    width = max(record_id_index, HN_END - 1) + 1
    for row_number, row in enumerate(rows[1:], start=2):
        padded = list(row) + [""] * (width - len(row))
        record_id = str(padded[record_id_index] or "").strip()
        if not record_id:
            skipped += 1
            continue
        payload = {"Id": record_id}
        for index, field in resolved:
            payload[field["name"]] = _convert(padded[index], field)
        send_ue = payload.get(ue_stop_name) is False
        send_shita = payload.get(shita_stop_name) is False
        if send_ue:
            ue_mail_count += 1
        if send_shita:
            shita_mail_count += 1
        updates.append({
            "row": row_number,
            "engineer": str(padded[engineer_index] or "").strip(),
            "ue_company": str(padded[ue_company_index] or "").strip(),
            "shita_company": str(padded[shita_company_index] or "").strip(),
            "payload": payload,
            "send_ue": send_ue,
            "send_shita": send_shita,
        })
    return updates, skipped, ue_mail_count, shita_mail_count



def _send_label(will_send):
    return "あり" if will_send else "なし"


def _address_html(text, multiline):
    raw = str(text or "").strip()
    if not raw or not multiline:
        return html.escape(str(text or ""))
    parts = [part.strip() for part in re.split(r"[,;\n]+", raw) if part.strip()]
    if len(parts) <= 1:
        return html.escape(parts[0] if parts else "")
    return "<br>".join(html.escape(part) for part in parts)


def _cell(text, rowspan=1, edge="", fill="#fff", color="#111", multiline=False):
    span = f' rowspan="{rowspan}"' if rowspan > 1 else ""
    border = "border:1px solid #666;"
    if edge == "top":
        border += "border-top:3px solid #222;"
    elif edge == "bottom":
        border += "border-bottom:3px solid #222;"
    elif edge == "group":
        border += "border-top:3px solid #222;border-bottom:3px solid #222;"
    return (
        f"<td{span} style=\"{border}padding:6px;text-align:left;"
        f"vertical-align:middle;word-break:break-all;background:{fill};color:{color}\">"
        f"{_address_html(text, multiline)}</td>"
    )


def _head(text, rowspan=1):
    span = f' rowspan="{rowspan}"' if rowspan > 1 else ""
    return (
        f"<th{span} style=\"border:1px solid #666;border-bottom:3px solid #222;"
        "padding:6px;text-align:left;vertical-align:middle;background:#e8e8e8\">"
        f"{html.escape(text)}</th>"
    )


def _fetch_mail_addresses(sf, record_ids):
    cleaned = []
    for record_id in record_ids:
        text = str(record_id or "").strip()
        if re.fullmatch(r"[A-Za-z0-9]{15,18}", text):
            cleaned.append(text)
    found = {}
    fields = (
        "Id, ueentyo_email__c, ueentyo_ccemail__c, "
        "shitaentyo_email__c, shitaentyo_ccemail__c"
    )
    for start in range(0, len(cleaned), 100):
        chunk = cleaned[start:start + 100]
        id_list = ", ".join(f"'{item}'" for item in chunk)
        soql = f"SELECT {fields} FROM kado__c WHERE Id IN ({id_list})"
        result = sf.query_all(soql)
        for rec in result.get("records") or []:
            record_id = rec.get("Id") or ""
            found[record_id] = rec
            if len(record_id) >= 15:
                found[record_id[:15]] = rec
    return found


def _attach_mail_addresses(sf, updates):
    addresses = _fetch_mail_addresses(
        sf, [item["payload"].get("Id") for item in updates]
    )
    for item in updates:
        record_id = str(item["payload"].get("Id") or "")
        rec = addresses.get(record_id) or addresses.get(record_id[:15]) or {}
        item["ue_to"] = rec.get("ueentyo_email__c") or ""
        item["ue_cc"] = rec.get("ueentyo_ccemail__c") or ""
        item["shita_to"] = rec.get("shitaentyo_email__c") or ""
        item["shita_cc"] = rec.get("shitaentyo_ccemail__c") or ""


def _direction_style(will_send, active_fill):
    if will_send:
        return active_fill, "#111"
    return "#e4e4e4", "#8a8a8a"


def _mail_confirm_table(updates):
    rows = [
        "<tr>"
        + _head("上企業名")
        + _head("下企業名")
        + _head("技術者名")
        + _head("メール種別")
        + _head("延長確認メール")
        + _head("TO")
        + _head("CC")
        + "</tr>"
    ]
    for item in updates:
        send_ue = bool(item.get("send_ue"))
        send_shita = bool(item.get("send_shita"))
        group_fill, group_color = _direction_style(
            send_ue or send_shita, "#fff"
        )
        ue_fill, ue_color = _direction_style(send_ue, "#e5f6e8")
        shita_fill, shita_color = _direction_style(send_shita, "#fff0e0")
        rows.append(
            "<tr>"
            + _cell(item.get("ue_company"), 2, "group", group_fill, group_color)
            + _cell(item.get("shita_company"), 2, "group", group_fill, group_color)
            + _cell(item.get("engineer"), 2, "group", group_fill, group_color)
            + _cell("上向け", edge="top", fill=ue_fill, color=ue_color)
            + _cell(_send_label(send_ue), edge="top", fill=ue_fill, color=ue_color)
            + _cell(item.get("ue_to"), edge="top", fill=ue_fill, color=ue_color, multiline=send_ue)
            + _cell(item.get("ue_cc"), edge="top", fill=ue_fill, color=ue_color, multiline=send_ue)
            + "</tr>"
        )
        rows.append(
            "<tr>"
            + _cell("下向け", edge="bottom", fill=shita_fill, color=shita_color)
            + _cell(_send_label(send_shita), edge="bottom", fill=shita_fill, color=shita_color)
            + _cell(item.get("shita_to"), edge="bottom", fill=shita_fill, color=shita_color, multiline=send_shita)
            + _cell(item.get("shita_cc"), edge="bottom", fill=shita_fill, color=shita_color, multiline=send_shita)
            + "</tr>"
        )
    return (
        "<table border=\"1\" cellspacing=\"0\" cellpadding=\"6\" "
        "style=\"border-collapse:collapse;width:100%;border:2px solid #222\">"
        "<thead>" + rows[0] + "</thead><tbody>"
        + "".join(rows[1:])
        + "</tbody></table>"
    )


def _confirm_mail_send(updates):
    from IPython.display import HTML, display
    from google.colab import output

    table = _mail_confirm_table(updates)
    display(HTML("<h3>メール送付の確認</h3>" + table))
    script = """
const tableHtml = TABLE_JSON;
new Promise((resolve) => {
  const wrap = document.createElement('div');
  wrap.style.cssText = 'position:fixed;inset:0;background:rgba(0,0,0,.45);z-index:99999;display:flex;align-items:center;justify-content:center;padding:24px;box-sizing:border-box;';
  const panel = document.createElement('div');
  panel.style.cssText = 'background:#fff;color:#111;max-width:1200px;width:100%;max-height:80vh;overflow:auto;padding:20px;border-radius:8px;font-family:sans-serif;';
  panel.innerHTML = '<h2 style="margin:0 0 12px">メール送付の確認</h2>' + tableHtml +
    '<p style="margin:12px 0">この内容で延長確認メールを送信します。</p>' +
    '<div style="display:flex;gap:8px;justify-content:flex-end">' +
    '<button id="mail-cancel" type="button">キャンセル</button>' +
    '<button id="mail-ok" type="button">送信する</button></div>';
  wrap.appendChild(panel);
  document.body.appendChild(wrap);
  panel.querySelector('#mail-ok').onclick = () => { wrap.remove(); resolve(true); };
  panel.querySelector('#mail-cancel').onclick = () => { wrap.remove(); resolve(false); };
})
""".replace("TABLE_JSON", json.dumps(table))
    try:
        return bool(output.eval_js(script))
    except Exception as exc:
        print(f"確認画面を表示できませんでした: {exc}")
        return False


def _apply(sf, updates):
    success = 0
    failures = []
    success_ids = []
    chunk_size = 100
    for start in range(0, len(updates), chunk_size):
        chunk_items = updates[start:start + chunk_size]
        results = sf.bulk.kado__c.update(
            [item["payload"] for item in chunk_items],
            batch_size=chunk_size,
            use_serial=True,
        )
        for item, result in zip(chunk_items, results):
            if result.get("success"):
                success += 1
                success_ids.append(item["payload"].get("Id"))
                continue
            errors = result.get("errors") or []
            message = " / ".join(
                error.get("message", str(error)) if isinstance(error, dict) else str(error)
                for error in errors
            ) or str(result)
            failures.append(f"{item['row']}行目 {item['payload'].get('Id')}: {message}")
    return success, failures, success_ids


def main():
    worksheet_name = str(_worksheet_name).strip()
    print(f"対象シート: {worksheet_name}")
    print(f"スプレッドシート: {SPREADSHEET_URL}")
    sf = setup_salesforce()
    described = _describe_fields(sf)
    gc = setup_google_sheets()
    rows = _read_sheet(gc, worksheet_name)
    updates, skipped, ue_mail_count, shita_mail_count = _build_updates(rows, described)
    print(f"更新対象: {len(updates)}件（レコードIDなし {skipped}件は除外）")
    print(f"上延長確認メール対象: {ue_mail_count}件")
    print(f"下延長確認メール対象: {shita_mail_count}件")
    if not updates:
        print("更新する行がありません。")
        return
    if not _apply_updates:
        print("「反映を実行する」がオフのため、書き込みもメール送信もしません。")
        print("先頭3件:")
        for item in updates[:3]:
            print(item["row"], item["payload"], "上メール" if item["send_ue"] else "", "下メール" if item["send_shita"] else "")
        return
    success, failures, success_ids = _apply(sf, updates)
    print(f"反映完了: {success}件")
    if failures:
        print(f"失敗: {len(failures)}件")
        for line in failures[:20]:
            print(line)
    sent_ids = {record_id for record_id in success_ids if record_id}
    mail_rows = [item for item in updates if item["payload"].get("Id") in sent_ids]
    pending = [item for item in mail_rows if item["send_ue"] or item["send_shita"]]
    if not pending:
        print("メール送付対象はありません。")
    else:
        _attach_mail_addresses(sf, mail_rows)
        if not _confirm_mail_send(mail_rows):
            print("メール送信をキャンセルしました。稼働レコードへの反映は完了しています。")
        else:
            ue_ok, shita_ok, mail_lines = _send_extension_mails(sf, updates, success_ids)
            print(f"上延長確認メール送信: {ue_ok}件")
            print(f"下延長確認メール送信: {shita_ok}件")
            for line in mail_lines:
                print(line)
    print(f"シート: {SPREADSHEET_URL}")


main()
